In [ ]:
from download_data import download_file, extract_archive


In [2]:
url_spch_1 = "https://openslr.elda.org/resources/12/dev-clean.tar.gz"
file_name_spch_1 = "dev-clean.tar.gz"
url_spch_2 = "https://openslr.elda.org/resources/12/test-clean.tar.gz"
file_name_spch_2 = "test-clean.tar.gz"
url_noise = "https://my-bucket-a8b4b49c25c811ee9a7e8bba05fa24c7.s3.amazonaws.com/wham_noise.zip"
file_name_noise = "wham_noise.zip"

In [3]:
download_file(url_spch_1, file_name_spch_1)

dev-clean.tar.gz:   0%|          | 0.00/322M [00:00<?, ?B/s]

File 'dev-clean.tar.gz' downloaded successfully.


In [4]:
download_file(url_spch_2, file_name_spch_2)

test-clean.tar.gz:   0%|          | 0.00/331M [00:00<?, ?B/s]

File 'test-clean.tar.gz' downloaded successfully.


In [5]:
download_file(url_noise, file_name_noise)

wham_noise.zip:   0%|          | 0.00/16.9G [00:00<?, ?B/s]

File 'wham_noise.zip' downloaded successfully.


In [6]:
extract_archive("wham_noise.zip")

Extracting:   0%|          | 0/28017 [00:00<?, ?file/s]

Extracted 'wham_noise.zip' to 'wham_noise'


In [7]:
extract_archive("dev-clean.tar.gz")

Extracting:   0%|          | 0/2943 [00:00<?, ?file/s]

Extracted 'dev-clean.tar.gz' to 'dev-clean.tar'


In [8]:
extract_archive("test-clean.tar.gz")

Extracting:   0%|          | 0/2840 [00:00<?, ?file/s]

Extracted 'test-clean.tar.gz' to 'test-clean.tar'


In [1]:
from loaders import get_loaders
from IPython.display import Audio 

In [2]:
train_loader, val_loader, test_loader = get_loaders(speech_dirs=["dataset/dev-clean/LibriSpeech/dev-clean", "/home/ys/diploma/dataset/test-clean/LibriSpeech/test-clean"],
                                                    noise_dir="/home/ys/diploma/dataset/wham_noise/wham_noise",
                                                    batch_size=1,
                                                    padding_strategy=None)

In [3]:
sample_batch = next(iter(train_loader))
mixed_waveforms, speech_waveforms = sample_batch

for i, (mixed_waveform, speech_waveform) in enumerate(zip(mixed_waveforms, speech_waveforms)):
    print(f"--------------------\nsample {i} from batch")
    print(f"Input audio {i + 1}:")
    display(Audio(mixed_waveform.numpy(), rate=train_loader.dataset.sample_rate))
    print(f"Target audio {i + 1}:")
    display(Audio(speech_waveform.numpy(), rate=train_loader.dataset.sample_rate))

--------------------
sample 0 from batch
Input audio 1:


Target audio 1:


In [31]:
import torch.nn as nn
import torch
import torchaudio.functional as F
import torchaudio.transforms as T

class SiSDRLoss(nn.Module):
    def __init__(self, eps: float = 1e-9):
        """

        Args:
            eps (float): eps for stabibiluty calculations. Defaults to 1e-9.
        """
        super().__init__()
        self.eps = eps


    def forward(self, output, target):

        alpha = torch.sum(output * target, dim=-1,keepdim=True) / torch.norm(target, dim=-1)**2 

        proj = alpha * target

        proj_norm = torch.norm(proj, dim=-1)
        diff_norm = torch.norm((proj - output), dim=-1)

        return  -(10 * (torch.log10(proj_norm**2 / (diff_norm**2 + self.eps )))).mean()

class MultiResolutionLoss(nn.Module):
    def __init__(self, n_ftts: list = [256, 512, 1025, 2096]):
        super().__init__()
        self.nftts = n_ftts
    def forward(self, clean, enchanced):

        return torch.mean(torch.stack([nn.functional.l1_loss(T.Spectrogram(n_fft=n_fft, 
                                                                           power=1.0,
                                                                           normalized=True)(clean).log1p(),
                                                           T.Spectrogram(n_fft=n_fft, 
                                                                         power=1.0,
                                                                         normalized=True)(enchanced).log1p()) for n_fft in self.nftts]))

mrsl = MultiResolutionLoss()
sisdr = SiSDRLoss()

mrsl(mixed_waveforms[:, 0 ,None,...], speech_waveforms[:, 0 ,None,...]), sisdr(mixed_waveforms[:, 0 ,None,...], speech_waveforms[:, 0 ,None,...]) 

(tensor(0.0009), tensor(-17.4629))